# Debug and Evaluate: Measuring Retrieval

Level 3: RAG and Retrieval Engineering.

**Problems in this notebook**

- Problem 25: Find Low-Confidence Retrievals
- Problem 29: Sample an Evaluation Set by Category

**How to use this notebook**

1. Read the problem and say your plan aloud before you type anything.
2. Paste the AI prompt given (or write your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `04_debug_and_evaluate_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 25: Find Low-Confidence Retrievals

**The situation**

A customer says the assistant gives poor answers. Before blaming the model, you want to see which questions got weak search results, or none at all.

**Your task**

Write `low_confidence_queries(results, threshold)`. Return `(query_id, reason)` for each query whose search found nothing (`"no results"`) or whose best score is below the threshold (`"low score"`).

**Example**

`[{"query_id": "q1", "scores": [0.2]}]`, `threshold=0.5` -> `[("q1", "low score")]`

**Ask before coding**

- Are scores from different retrievers comparable to one threshold?
- Does a score exactly equal to the threshold count as OK?
- Do you also want the query text and top chunks for the report?

**Plan to say aloud**

1. Loop over the results in order.
2. Skip rows with no `query_id`; we cannot report them by name.
3. If the scores list is missing or empty, flag "no results".
4. Otherwise, if the best score is below the threshold, flag "low score".
5. Return the list of flagged pairs.

### Prompt for your AI assistant

```text
Write a Python function low_confidence_queries(results: list[dict], threshold: float) -> list[tuple[str, str]].
Each result has "query_id" and "scores" (a list of floats, may be missing or empty).
Rules:
- skip rows with no query_id
- empty or missing scores -> (query_id, "no results")
- max(scores) < threshold -> (query_id, "low score"); equal to threshold is fine
- keep input order
Show only the function.
```

**Check the AI's code for**

- Calls `max(scores)` before checking for an empty list, which raises `ValueError`.
- Checks the first score or the average instead of the best score.
- Uses `<=` so a score right at the threshold gets flagged.
- Returns only IDs, losing the reason that tells you what to fix.

In [ ]:
def low_confidence_queries(results: list[dict], threshold: float) -> list[tuple[str, str]]:
    """Flag queries with no results or a weak best score.

    Rules:
    - skip rows with no query_id
    - missing/empty scores -> (query_id, "no results")
    - max score < threshold -> (query_id, "low score"); equal is fine
    - keep input order
    """
    raise NotImplementedError

In [ ]:
cases = [
    (([{"query_id": "q1", "scores": [0.2]}], 0.5), [("q1", "low score")]),
    (([{"query_id": "q2", "scores": [0.1, 0.9]}], 0.5), []),
    (([{"query_id": "q3", "scores": []}, {"query_id": "q4"}], 0.5),
     [("q3", "no results"), ("q4", "no results")]),
    # exactly at the threshold is fine
    (([{"query_id": "q5", "scores": [0.5]}], 0.5), []),
    # mixed, plus a row with no id
    (([{"query_id": "a", "scores": [0.7]}, {"scores": [0.0]},
       {"query_id": "b", "scores": [0.3, 0.1]}], 0.5), [("b", "low score")]),
]
run_checks(low_confidence_queries, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def low_confidence_queries_reference(results: list[dict], threshold: float) -> list[tuple[str, str]]:
    flagged = []
    for result in results:
        query_id = result.get("query_id")
        if not query_id:
            continue
        scores = result.get("scores") or []
        if not scores:
            flagged.append((query_id, "no results"))
        elif max(scores) < threshold:
            flagged.append((query_id, "low score"))
    return flagged


assert run_checks(low_confidence_queries_reference, cases)

**Explain it in 30 seconds**

> "When answers are bad, I first check what was retrieved. For each query I flag it if nothing came back, or if even the best match scored below the threshold. I return the reason with the ID, because "no results" and "low score" lead to different fixes. I would also confirm the threshold makes sense, since raw search scores are often not calibrated."

**Edge cases to mention**

- An empty scores list, which must not crash `max`.
- Scores from two retrievers on different scales sharing one threshold.
- A score exactly at the threshold.
- Rows with no query ID from a broken log line.

**If they ask "how would this work in production?"**

- Log the query, filters, top chunks, scores, and final answer for every request so you can replay bad cases.
- Keep a set of known good questions and rerun them after every index or model change.
- Show the share of low-confidence queries per tenant on a dashboard and alert on jumps.

## Problem 29: Sample an Evaluation Set by Category

**The situation**

A customer wants to test the assistant across HR, IT, and finance questions. Random sampling from their logs gives mostly IT questions and almost no finance, so the test misses whole areas.

**Your task**

Write `sample_per_category(examples, limit)` that keeps at most `limit` examples from each category. The same input must always give the same output.

**Example**

hr, hr, it with `limit=1` -> the first hr and the it example

**Ask before coding**

- Which field holds the category, and what if it is missing?
- Should the sample be the same every run, or random with a seed?
- Should sensitive examples be removed first?

**Plan to say aloud**

1. Reject a negative limit.
2. Group examples by category in the order categories first appear. Missing category goes to "unknown".
3. Take the first `limit` examples from each group.
4. Return them group by group.

### Prompt for your AI assistant

```text
Write a Python function sample_per_category(examples: list[dict], limit: int) -> list[dict].
Rules:
- raise ValueError if limit < 0
- group by example["category"]; a missing category counts as "unknown"
- keep the first `limit` examples of each category, in input order
- output is grouped by category, categories in first-seen order
- deterministic: no random module
Standard library only. Show only the function.
```

**Check the AI's code for**

- Uses `random.sample`, so the eval set changes every run and scores cannot be compared.
- Crashes with `KeyError` on an example with no category instead of putting it in "unknown".
- Sorts categories alphabetically when you asked for first-seen order.
- Returns a dict of groups instead of one flat list.

In [ ]:
def sample_per_category(examples: list[dict], limit: int) -> list[dict]:
    """Keep at most `limit` examples per category, deterministically.

    Rules:
    - ValueError if limit < 0
    - missing category counts as "unknown"
    - first `limit` examples of each category, in input order
    - output grouped by category, categories in first-seen order
    """
    raise NotImplementedError

In [ ]:
HR1, HR2 = {"id": 1, "category": "hr"}, {"id": 2, "category": "hr"}
IT1 = {"id": 3, "category": "it"}
NONE = {"id": 4}
cases = [
    (([HR1, HR2, IT1], 1), [HR1, IT1]),
    # output is grouped by category, first-seen order
    (([HR1, IT1, HR2], 2), [HR1, HR2, IT1]),
    # missing category goes to "unknown"
    (([NONE, HR1], 1), [NONE, HR1]),
    (([HR1, IT1], 0), []),
    (([], 3), []),
]
run_checks(sample_per_category, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def sample_per_category_reference(examples: list[dict], limit: int) -> list[dict]:
    if limit < 0:
        raise ValueError("limit must be 0 or more")
    groups: dict[str, list[dict]] = {}
    for example in examples:
        groups.setdefault(example.get("category", "unknown"), []).append(example)
    sampled = []
    for group in groups.values():
        sampled.extend(group[:limit])
    return sampled


assert run_checks(sample_per_category_reference, cases)

**Explain it in 30 seconds**

> "I group the examples by category and keep the first few from each, so small areas like finance are covered as well as big ones like IT. Missing categories go into an "unknown" bucket instead of crashing. The result is the same every run, so I can compare scores between versions. A small balanced test set beats a large lopsided one."

**Edge cases to mention**

- Examples with no category.
- A category with fewer examples than the limit.
- A limit of zero or a negative limit.
- Examples that contain personal or sensitive data.

**If they ask "how would this work in production?"**

- Use seeded random sampling inside each category so you get variety but can still repeat it.
- Version the eval set and store which version produced each score.
- Report coverage by department, language, and document type, and filter sensitive examples first.

## Wrap-up

- Both are about evidence: find where retrieval is weak, and build a fair set of questions to test it.
- Return results that explain themselves (a reason, a stable sample) so people can act on them.
- Deterministic outputs make before-and-after comparisons honest.